# Chapter 15: Diversity, Coverage, and Evidence Budgets

Whole-unit selection under a character budget.

In [ ]:
import sys
from pathlib import Path

ROOT = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "content" / "books" / "retrieval" / "_index.md").is_file():
        ROOT = candidate
        break
if ROOT is None:
    raise FileNotFoundError("repository root not found")

NOTEBOOK_DIR = ROOT / "notebooks" / "books" / "retrieval"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import _support
ROOT = _support.repo_root()
print("repo root:", ROOT)


In [ ]:
from document_identity import Collection
from evidence_units import build_units
from evidence_pipeline import identity, method_candidates, select

collection = Collection.load(ROOT / "examples" / "fixtures" / "part2-v1.json")
units = build_units(collection, "part2", "sections")
a = next(u for u in units if u["source_id"] == "toy:blog")
b = next(u for u in units if u["source_id"] == "toy:spec")
c = next(u for u in units if u["document_id"] == "b-focused")

rows = method_candidates(
    [{**a, "score": 3.0}, {**b, "score": 2.0}, {**c, "score": 1.0}],
    "bm25", "part2", "hand-test",
)
print("candidates:", len(rows))

In [ ]:
# Whole-unit packing obeys the character budget and separator cost.
limit = len(a["text"]) + len(b["text"]) + 2
packed = select(rows, characters=limit)
print("packed units:", len(packed["selected"]), "chars:", packed["characters"])
assert len(packed["selected"]) == 2
assert packed["characters"] == limit

In [ ]:
# MMR can prefer a complementary unit when the budget allows a choice.
vectors = {a["passage_id"]: [1, 0], b["passage_id"]: [1, 0], c["passage_id"]: [0, 1]}
mmr = select(rows, max_units=2, characters=1000, vectors=vectors)
print("MMR selected:", [identity(x["unit"]) for x in mmr["selected"]])
assert [identity(x["unit"]) for x in mmr["selected"]] == [identity(a), identity(c)]

In [ ]:
# Under a tight cap, a critical unit may not fit; diversity cannot recover it.
critical = c
small = a
tight = len(small["text"])
budgeted = select(rows, characters=tight)
budgeted_ids = {x["unit"]["passage_id"] for x in budgeted["selected"]}
print("budgeted selected:", budgeted_ids)
assert critical["passage_id"] not in budgeted_ids

from evidence_pipeline import document_metrics
metrics = document_metrics([critical["document_id"]], budgeted["selected"])
print("budgeted recall:", metrics["recall@3"])
assert metrics["recall@3"] == 0.0

In [ ]:
# With only one feasible packet, MMR and character packing agree (no diversity win here).
pack_only = select(rows, characters=tight, vectors=None)
mmr_only = select(rows, characters=tight, vectors=vectors)
pack_ids = [identity(x["unit"]) for x in pack_only["selected"]]
mmr_ids = [identity(x["unit"]) for x in mmr_only["selected"]]
print("pack:", pack_ids)
print("mmr: ", mmr_ids)
assert pack_ids == mmr_ids

## Observed versus not established
Observed: whole-unit feasibility, MMR mechanism, and non-recovery under tight caps. Not established: optimal budget, diversity value or downstream helpfulness.